# Transformer 모델 실습
- 교재 torchtext가 Colab 최신 환경에서 정상 작동하지 않아서 테스트용 더미 데이터로 코드 수정했습니다!


In [1]:
!pip install torchtext portalocker datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 28.3 MB/s eta 0:00:00


In [2]:
# 7.2 데이터세트 다운로드 및 전처리

!pip install portalocker spacy

!python -m spacy download de_core_news_sm
!python -m spacy download en_core_web_sm

import spacy

SRC_LANGUAGE = "de"
TGT_LANGUAGE = "en"
UNK_IDX, PAD_IDX, BOS_IDX, EOS_IDX = 0, 1, 2, 3
special_symbols = ["<unk>", "<pad>", "<bos>", "<eos>"]

nlp_de = spacy.load("de_core_news_sm")
nlp_en = spacy.load("en_core_web_sm")

def get_tokenizer_spacy(language):
    if language == "de_core_news_sm":
        return lambda text: [tok.text for tok in nlp_de(text)]
    elif language == "en_core_web_sm":
        return lambda text: [tok.text for tok in nlp_en(text)]
    else:
        raise ValueError("unknown language")

def generate_tokens(text_iter, language):
    language_index = {SRC_LANGUAGE: 0, TGT_LANGUAGE: 1}
    for text in text_iter:
        yield token_transform[language](text[language_index[language]])

token_transform = {
    SRC_LANGUAGE: get_tokenizer_spacy("de_core_news_sm"),
    TGT_LANGUAGE: get_tokenizer_spacy("en_core_web_sm"),
}
print("Token Transform:")
print(token_transform)

class SimpleVocab:
    def __init__(self, stoi, specials):
        self.stoi = dict(stoi)
        self.itos = [""] * len(stoi)
        for token, idx in self.stoi.items():
            if idx >= len(self.itos):
                self.itos.extend([""] * (idx - len(self.itos) + 1))
            self.itos[idx] = token
        self.default_index = None
        self.specials = specials

    def __call__(self, tokens):
        return [self.stoi.get(tok, self.default_index if self.default_index is not None else 0) for tok in tokens]

    def set_default_index(self, idx):
        self.default_index = idx

    def lookup_tokens(self, indices):
        return [self.itos[idx] if 0 <= idx < len(self.itos) else self.specials[0] for idx in indices]

dummy_data = [
    ("Eine Gruppe von Menschen steht vor einem Gebäude .", "A group of people stands in front of a building ."),
    ("Ein Hund rennt über das Feld .", "A dog is running across the field ."),
    ("Ein Mann spielt Gitarre auf der Straße .", "A man is playing guitar on the street ."),
    ("Ein Mädchen liest ein Buch im Park .", "A girl reads a book in the park ."),
]

def build_vocab_from_iterator_like(text_iter, language):
    stoi = {}
    idx = 0
    for sym in special_symbols:
        stoi[sym] = idx
        idx += 1
    for tokens in generate_tokens(text_iter, language):
        for tok in tokens:
            if tok not in stoi:
                stoi[tok] = idx
                idx += 1
    return SimpleVocab(stoi, special_symbols)

vocab_transform = {}
for language in [SRC_LANGUAGE, TGT_LANGUAGE]:
    text_iter = dummy_data
    vocab_transform[language] = build_vocab_from_iterator_like(text_iter, language)

for language in [SRC_LANGUAGE, TGT_LANGUAGE]:
    vocab_transform[language].set_default_index(UNK_IDX)
print("Vocab Transform:")
print(vocab_transform)


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.6/14.6 MB 90.1 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('de_core_news_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 32.5 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
Token Transform:
{'de': <function get_tokenizer_spacy.<locals>.<lambda> at 0x7d6594dbd940>, 'en': <function get_tokenizer_spacy.<locals>.<lambda> at 0x7d6594dbf060>}

In [3]:
# 7.3 트랜스포머 모델 구성

import math
import torch
from torch import nn

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)
        position = torch.arange(max_len).unsqueeze(1)
        div_term = torch.exp(
            torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model)
        )
        pe = torch.zeros(max_len, 1, d_model)
        pe[:, 0, 0::2] = torch.sin(position * div_term)
        pe[:, 0, 1::2] = torch.cos(position * div_term)
        self.register_buffer("pe", pe)

    def forward(self, x):
        x = x + self.pe[: x.size(0)]
        return self.dropout(x)

class TokenEmbedding(nn.Module):
    def __init__(self, vocab_size, emb_size):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb_size)
        self.emb_size = emb_size

    def forward(self, tokens):
        return self.embedding(tokens.long()) * math.sqrt(self.emb_size)

class Seq2SeqTransformer(nn.Module):
    def __init__(
        self,
        num_encoder_layers,
        num_decoder_layers,
        emb_size,
        max_len,
        nhead,
        src_vocab_size,
        tgt_vocab_size,
        dim_feedforward,
        dropout=0.1,
    ):
        super().__init__()
        self.src_tok_emb = TokenEmbedding(src_vocab_size, emb_size)
        self.tgt_tok_emb = TokenEmbedding(tgt_vocab_size, emb_size)
        self.positional_encoding = PositionalEncoding(
            d_model=emb_size, max_len=max_len, dropout=dropout
        )
        self.transformer = nn.Transformer(
            d_model=emb_size,
            nhead=nhead,
            num_encoder_layers=num_encoder_layers,
            num_decoder_layers=num_decoder_layers,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
        )
        self.generator = nn.Linear(emb_size, tgt_vocab_size)

    def forward(
        self,
        src,
        tgt,
        src_mask,
        tgt_mask,
        src_padding_mask,
        tgt_padding_mask,
        memory_key_padding_mask,
    ):
        src_emb = self.positional_encoding(self.src_tok_emb(src))
        tgt_emb = self.positional_encoding(self.tgt_tok_emb(tgt))
        outs = self.transformer(
            src=src_emb,
            tgt=tgt_emb,
            src_mask=src_mask,
            tgt_mask=tgt_mask,
            memory_mask=None,
            src_key_padding_mask=src_padding_mask,
            tgt_key_padding_mask=tgt_padding_mask,
            memory_key_padding_mask=memory_key_padding_mask
        )
        return self.generator(outs)

    def encode(self, src, src_mask):
        return self.transformer.encoder(
            self.positional_encoding(self.src_tok_emb(src)), src_mask
        )

    def decode(self, tgt, memory, tgt_mask):
        return self.transformer.decoder(
            self.positional_encoding(self.tgt_tok_emb(tgt)), memory, tgt_mask
        )

In [4]:
# 7.4 트랜스포머 모델 구조

from torch import optim

BATCH_SIZE = 4
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

model = Seq2SeqTransformer(
    num_encoder_layers=1,
    num_decoder_layers=1,
    emb_size=256,
    max_len=256,
    nhead=4,
    src_vocab_size=len(vocab_transform[SRC_LANGUAGE].stoi),
    tgt_vocab_size=len(vocab_transform[TGT_LANGUAGE].stoi),
    dim_feedforward=256,
).to(DEVICE)

criterion = nn.CrossEntropyLoss(ignore_index=PAD_IDX).to(DEVICE)
optimizer = optim.Adam(model.parameters())

for main_name, main_module in model.named_children():
    print(main_name)
    for sub_name, sub_module in main_module.named_children():
        print("ㄴ", sub_name)

/usr/local/lib/python3.12/dist-packages/torch/nn/modules/transformer.py:392: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  warnings.warn(


src_tok_emb
ㄴ embedding
tgt_tok_emb
ㄴ embedding
positional_encoding
ㄴ dropout
transformer
ㄴ encoder
ㄴ decoder
generator


In [5]:
# 7.5 배치 데이터 생성

from torch.utils.data import DataLoader
from torch.nn.utils.rnn import pad_sequence

def sequential_transforms(*transforms):
    def func(txt_input):
        for transform in transforms:
            txt_input = transform(txt_input)
        return txt_input
    return func

def input_transform(token_ids):
    return torch.cat(
        (torch.tensor([BOS_IDX]), torch.tensor(token_ids), torch.tensor([EOS_IDX]))
    )

def collator(batch):
    src_batch, tgt_batch = [], []
    for src_sample, tgt_sample in batch:
        src_batch.append(text_transform[SRC_LANGUAGE](src_sample))
        tgt_batch.append(text_transform[TGT_LANGUAGE](tgt_sample))
    src_batch = pad_sequence(src_batch, padding_value=PAD_IDX)
    tgt_batch = pad_sequence(tgt_batch, padding_value=PAD_IDX)
    return src_batch, tgt_batch

text_transform = {}
for language in [SRC_LANGUAGE, TGT_LANGUAGE]:
    text_transform[language] = sequential_transforms(
        token_transform[language], vocab_transform[language], input_transform
    )

data_iter = dummy_data
dataloader = DataLoader(data_iter, batch_size=BATCH_SIZE, collate_fn=collator)
source_tensor, target_tensor = next(iter(dataloader))

print("(source, target:)")
print(next(iter(iter(dummy_data))))
print("source_batch:", source_tensor.shape)
print(source_tensor)
print("target_batch:", target_tensor.shape)
print(target_tensor)

(source, target:)
('Eine Gruppe von Menschen steht vor einem Gebäude .', 'A group of people stands in front of a building .')
source_batch: torch.Size([11, 4])
tensor([[ 2,  2,  2,  2],
        [ 4, 13, 13, 13],
        [ 5, 14, 19, 25],
        [ 6, 15, 20, 26],
        [ 7, 16, 21, 27],
        [ 8, 17, 22, 28],
        [ 9, 18, 23, 29],
        [10, 12, 24, 30],
        [11,  3, 12, 12],
        [12,  1,  3,  3],
        [ 3,  1,  1,  1]])
target_batch: torch.Size([13, 4])
tensor([[ 2,  2,  2,  2],
        [ 4,  4,  4,  4],
        [ 5, 14, 20, 25],
        [ 6, 15, 15, 26],
        [ 7, 16, 21, 11],
        [ 8, 17, 22, 27],
        [ 9, 18, 23,  9],
        [10, 19, 18, 18],
        [ 6, 13, 24, 28],
        [11,  3, 13, 13],
        [12,  1,  3,  3],
        [13,  1,  1,  1],
        [ 3,  1,  1,  1]])


In [6]:
# 7.6 어텐션 마스크 생성

def generate_square_subsequent_mask(s):
    mask = (torch.triu(torch.ones((s, s), device=DEVICE)) == 1).transpose(0, 1)
    mask = (
        mask.float()
        .masked_fill(mask == 0, float("-inf"))
        .masked_fill(mask == 1, float(0.0))
    )
    return mask

def create_mask(src, tgt):
    src_seq_len = src.shape[0]
    tgt_seq_len = tgt.shape[0]
    tgt_mask = generate_square_subsequent_mask(tgt_seq_len)
    src_mask = torch.zeros((src_seq_len, src_seq_len), device=DEVICE).type(torch.bool)
    src_padding_mask = (src == PAD_IDX).transpose(0, 1)
    tgt_padding_mask = (tgt == PAD_IDX).transpose(0, 1)
    return src_mask, tgt_mask, src_padding_mask, tgt_padding_mask

target_input = target_tensor[:-1, :]
target_out = target_tensor[1:, :]

source_mask, target_mask, source_padding_mask, target_padding_mask = create_mask(
    source_tensor, target_input
)

print("source_mask:", source_mask.shape)
print(source_mask)
print("target_mask:", target_mask.shape)
print(target_mask)
print("source_padding_mask:", source_padding_mask.shape)
print(source_padding_mask)
print("target_padding_mask:", target_padding_mask.shape)
print(target_padding_mask)

source_mask: torch.Size([11, 11])
tensor([[False, False, False, False, False, False, False, False, False, False,
         False],
        [False, False, False, False, False, False, False, False, False, False,
         False],
        [False, False, False, False, False, False, False, False, False, False,
         False],
        [False, False, False, False, False, False, False, False, False, False,
         False],
        [False, False, False, False, False, False, False, False, False, False,
         False],
        [False, False, False, False, False, False, False, False, False, False,
         False],
        [False, False, False, False, False, False, False, False, False, False,
         False],
        [False, False, False, False, False, False, False, False, False, False,
         False],
        [False, False, False, False, False, False, False, False, False, False,
         False],
        [False, False, False, False, False, False, False, False, False, False,
         False],
      

In [7]:
# 7.7 모델 학습 및 평가

def run(model, optimizer, criterion, split):
    model.train() if split == "train" else model.eval()
    data_iter = dummy_data
    dataloader = DataLoader(data_iter, batch_size=BATCH_SIZE, collate_fn=collator)

    losses = 0
    for source_batch, target_batch in dataloader:
        source_batch = source_batch.to(DEVICE)
        target_batch = target_batch.to(DEVICE)

        target_input = target_batch[:-1, :]
        target_output = target_batch[1:, :]

        src_mask, tgt_mask, src_padding_mask, tgt_padding_mask = create_mask(
            source_batch, target_input
        )

        logits = model(
            src=source_batch,
            tgt=target_input,
            src_mask=src_mask,
            tgt_mask=tgt_mask,
            src_padding_mask=src_padding_mask,
            tgt_padding_mask=tgt_padding_mask,
            memory_key_padding_mask=src_padding_mask,
        )

        optimizer.zero_grad()
        loss = criterion(logits.reshape(-1, logits.shape[-1]), target_output.reshape(-1))
        if split == "train":
            loss.backward()
            optimizer.step()
        losses += loss.item()

    return losses / len(list(dataloader))

for epoch in range(5):
    train_loss = run(model, optimizer, criterion, "train")
    val_loss = run(model, optimizer, criterion, "valid")
    print(f"Epoch: {epoch+1}, Train loss: {train_loss:.3f}, Val loss: {val_loss:.3f}")

/usr/local/lib/python3.12/dist-packages/torch/nn/functional.py:6044: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  warnings.warn(


Epoch: 1, Train loss: 3.421, Val loss: 2.646
Epoch: 2, Train loss: 2.741, Val loss: 2.064
Epoch: 3, Train loss: 2.265, Val loss: 1.620
Epoch: 4, Train loss: 1.930, Val loss: 1.233
Epoch: 5, Train loss: 1.574, Val loss: 0.906


In [8]:
# 7.8 트랜스포머 모델 번역 결과

def greedy_decode(model, source_tensor, source_mask, max_len, start_symbol):
    source_tensor = source_tensor.to(DEVICE)
    source_mask = source_mask.to(DEVICE)

    memory = model.encode(source_tensor, source_mask)
    ys = torch.ones(1, 1).fill_(start_symbol).type(torch.long).to(DEVICE)
    for i in range(max_len - 1):
        memory = memory.to(DEVICE)
        target_mask = generate_square_subsequent_mask(ys.size(0))
        target_mask = target_mask.type(torch.bool).to(DEVICE)

        out = model.decode(ys, memory, target_mask)
        out = out.transpose(0, 1)
        prob = model.generator(out[:, -1])
        _, next_word = torch.max(prob, dim=1)
        next_word = next_word.item()

        ys = torch.cat(
            [ys, torch.ones(1, 1).type_as(source_tensor.data).fill_(next_word)], dim=0
        )
        if next_word == EOS_IDX:
            break

    return ys

def translate(model, source_sentence):
    model.eval()
    source_tensor = text_transform[SRC_LANGUAGE](source_sentence).view(-1, 1)
    num_tokens = source_tensor.shape[0]
    src_mask = (torch.zeros(num_tokens, num_tokens)).type(torch.bool)
    tgt_tokens = greedy_decode(
        model, source_tensor, src_mask, max_len=num_tokens + 5, start_symbol=BOS_IDX
    ).flatten()
    output = vocab_transform[TGT_LANGUAGE].lookup_tokens(list(tgt_tokens.cpu().numpy()))[1:-1]
    return " ".join(output)

output_oov = translate(model, "Eine Gruppe von Menschen steht vor einem Iglu .")
output = translate(model, "Eine Gruppe von Menschen steht vor einem Gebäude .")
print(output_oov)
print(output)

A group of a building .
A group of a building .


# GPT 모델 실습

In [9]:
from transformers import GPT2LMHeadModel

model = GPT2LMHeadModel.from_pretrained(pretrained_model_name_or_path = "gpt2")

for main_name, main_module in model.named_children():
  print(main_name)
  for sub_name, sub_module in main_module.named_children():
    print("ㄴ", sub_name)
    for ssub_name, ssub_module in sub_module.named_children():
      print("| ㄴ", ssub_name)
      for sssub_name, sssub_module in ssub_module.named_children():
        print("| | ㄴ", sssub_name)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

transformer
ㄴ wte
ㄴ wpe
ㄴ drop
ㄴ h
| ㄴ 0
| | ㄴ ln_1
| | ㄴ attn
| | ㄴ ln_2
| | ㄴ mlp
| ㄴ 1
| | ㄴ ln_1
| | ㄴ attn
| | ㄴ ln_2
| | ㄴ mlp
| ㄴ 2
| | ㄴ ln_1
| | ㄴ attn
| | ㄴ ln_2
| | ㄴ mlp
| ㄴ 3
| | ㄴ ln_1
| | ㄴ attn
| | ㄴ ln_2
| | ㄴ mlp
| ㄴ 4
| | ㄴ ln_1
| | ㄴ attn
| | ㄴ ln_2
| | ㄴ mlp
| ㄴ 5
| | ㄴ ln_1
| | ㄴ attn
| | ㄴ ln_2
| | ㄴ mlp
| ㄴ 6
| | ㄴ ln_1
| | ㄴ attn
| | ㄴ ln_2
| | ㄴ mlp
| ㄴ 7
| | ㄴ ln_1
| | ㄴ attn
| | ㄴ ln_2
| | ㄴ mlp
| ㄴ 8
| | ㄴ ln_1
| | ㄴ attn
| | ㄴ ln_2
| | ㄴ mlp
| ㄴ 9
| | ㄴ ln_1
| | ㄴ attn
| | ㄴ ln_2
| | ㄴ mlp
| ㄴ 10
| | ㄴ ln_1
| | ㄴ attn
| | ㄴ ln_2
| | ㄴ mlp
| ㄴ 11
| | ㄴ ln_1
| | ㄴ attn
| | ㄴ ln_2
| | ㄴ mlp
ㄴ ln_f
lm_head


In [10]:
# 7.10 GPT-2를 이용한 문장 생성
from transformers import pipeline

generator = pipeline(task="text-generation", model="gpt2")
outputs = generator(
    text_inputs = "Machine learning is",
    max_length=20,
    num_return_sequences=3,
    pad_token_id = generator.tokenizer.eos_token_id
)
print(outputs)

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Device set to use cuda:0
Truncation was not explicitly activated but `max_length` is provided a specific value, please use `truncation=True` to explicitly truncate examples to max length. Defaulting to 'longest_first' truncation strategy. If you encode pairs of sequences (GLUE-style) with the tokenizer you can select this strategy more precisely by providing a specific strategy to `truncation`.
Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[{'generated_text': "Machine learning is also part of the reason I decided to make this post. While the whole idea of learning is to learn something new, there is a lot of time invested in learning how to use it. I've learned a lot about the business of teaching and it's not just about learning how to do it. Learning is one of the key ways to earn money.\n\nI love teaching. I love teaching. I love learning. I love learning.\n\nIf you've ever done any of the above things, you know that learning is a lot of work. It's not just learning how to do something. It's learning how to do something that actually works.\n\nNow imagine that you've just been learning for 3 weeks. By that time you really have learned how to do something. You've learned to use your hand to push buttons. You've learned to use your body to bend and bend. You've learned to use your hands and legs to move your feet.\n\nYou've learned how to use your head to reach for objects. You've learned to use your hands to bend and b

In [11]:
import torch
from datasets import load_dataset
from transformers import AutoTokenizer
from torch.utils.data import DataLoader

cola = load_dataset("glue", "cola")
train_data = cola["train"]
valid_data = cola["validation"]
test_data = cola["test"]

tokenizer = AutoTokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token

device = "cuda" if torch.cuda.is_available() else "cpu"
batch_size = 16
epochs = 3

def collator(batch, tokenizer, device):
    texts = [item["sentence"] for item in batch]
    labels = [item["label"] for item in batch]

    tokenized = tokenizer(
        texts,
        padding="longest",
        truncation=True,
        return_tensors="pt"
    )

    input_ids = tokenized["input_ids"].to(device)
    attention_mask = tokenized["attention_mask"].to(device)
    labels = torch.tensor(labels, dtype=torch.long).to(device)

    return input_ids, attention_mask, labels

train_dataloader = DataLoader(
    train_data,
    batch_size=batch_size,
    shuffle=True,
    collate_fn=lambda x: collator(x, tokenizer, device)
)

valid_dataloader = DataLoader(
    valid_data,
    batch_size=batch_size,
    collate_fn=lambda x: collator(x, tokenizer, device)
)

test_dataloader = DataLoader(
    test_data,
    batch_size=batch_size,
    collate_fn=lambda x: collator(x, tokenizer, device)
)

print("Train Dataset Length :", len(train_data))
print("Valid Dataset Length :", len(valid_data))
print("Test Dataset Length :", len(test_data))

README.md: 0.00B [00:00, ?B/s]

cola/train-00000-of-00001.parquet:   0%|          | 0.00/251k [00:00<?, ?B/s]

cola/validation-00000-of-00001.parquet:   0%|          | 0.00/37.6k [00:00<?, ?B/s]

cola/test-00000-of-00001.parquet:   0%|          | 0.00/37.7k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/8551 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1043 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1063 [00:00<?, ? examples/s]

Train Dataset Length : 8551
Valid Dataset Length : 1043
Test Dataset Length : 1063


In [12]:
from torch import optim
from transformers import GPT2ForSequenceClassification

model = GPT2ForSequenceClassification.from_pretrained(
    pretrained_model_name_or_path="gpt2",
    num_labels=2
).to(device)

model.config.pad_token_id = model.config.eos_token_id

optimizer = optim.Adam(model.parameters(), lr=5e-5)

Some weights of GPT2ForSequenceClassification were not initialized from the model checkpoint at gpt2 and are newly initialized: ['score.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [13]:
import os
os.makedirs("./models", exist_ok=True)

In [14]:
import numpy as np
from torch import nn
import torch

def calc_accuracy(preds, labels):
    pred_flat = np.argmax(preds, axis=1).flatten()
    labels_flat = labels.flatten()
    return np.sum(pred_flat == labels_flat) / len(labels_flat)

def train(model, optimizer, dataloader):
    model.train()
    train_loss = 0.0

    for input_ids, attention_mask, labels in dataloader:
        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )

        loss = outputs.loss
        train_loss += loss.item()

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    return train_loss / len(dataloader)

def evaluation(model, dataloader):
    with torch.no_grad():
        model.eval()
        criterion = nn.CrossEntropyLoss()
        val_loss, val_accuracy = 0.0, 0.0

        for input_ids, attention_mask, labels in dataloader:
            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=labels
            )
            logits = outputs.logits

            loss = criterion(logits, labels)
            logits = logits.detach().cpu().numpy()
            label_ids = labels.cpu().numpy()
            accuracy = calc_accuracy(logits, label_ids)

            val_loss += loss
            val_accuracy += accuracy

        return val_loss / len(dataloader), val_accuracy / len(dataloader)

best_loss = 10000
for epoch in range(epochs):
    train_loss = train(model, optimizer, train_dataloader)
    val_loss, val_accuracy = evaluation(model, valid_dataloader)

    print(f"Epoch {epoch + 1}: Train Loss: {train_loss:.4f} Val Loss: {val_loss:.4f} Val Accuracy {val_accuracy:.4f}")

    if val_loss < best_loss:
        best_loss = val_loss
        torch.save(model.state_dict(), "./models/GPT2ForSequenceClassification.pt")
        print("Saved the model weights")

Epoch 1: Train Loss: 0.6139 Val Loss: 0.5759 Val Accuracy 0.6929
Saved the model weights
Epoch 2: Train Loss: 0.5258 Val Loss: 0.5335 Val Accuracy 0.7282
Saved the model weights
Epoch 3: Train Loss: 0.3756 Val Loss: 0.5509 Val Accuracy 0.7592


In [15]:
from transformers import GPT2ForSequenceClassification
import torch

model = GPT2ForSequenceClassification.from_pretrained(
    pretrained_model_name_or_path="gpt2",
    num_labels=2
).to(device)

model.config.pad_token_id = model.config.eos_token_id

model.load_state_dict(torch.load("./models/GPT2ForSequenceClassification.pt"))

test_loss, test_accuracy = evaluation(model, valid_dataloader)

print(f"Val Loss : {test_loss:.4f}")
print(f"Val Accuracy : {test_accuracy:.4f}")


Some weights of GPT2ForSequenceClassification were not initialized from the model checkpoint at gpt2 and are newly initialized: ['score.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Val Loss : 0.5335
Val Accuracy : 0.7282
